In [1]:
import os
# os.chdir('./back-end/')
!pwd


/Users/alex/code/mj/prism-editor/back-end


In [227]:
import gpt
from importlib import reload
reload(gpt)
from gpt import *

# Constraint.py testing

In [61]:
import gpt
from importlib import reload
reload(gpt)
from gpt import *
import constraints.constraint
from importlib import reload
reload(constraints.constraint)
from constraints.constraint import *

## Directly test constraint.py

In [64]:
pos_checker = POSChecker(['ADJ', 'ADJ', 'NOUN'], tokenizer)
# pos_checker2 = POSChecker(['ADV', 'VERB'], tokenizer)

# Create a ClassifiableConstraint with 'contains' mode
constraint = ClassifiableConstraint(pos_checker, 'contains', tokenizer)
# constraint2 = ClassifiableConstraint(pos_checker2, 'contains', tokenizer)

# constraints = [constraint, constraint2]
constraints = [constraint]

input = "When we want to go"
output = forward_search(
    input, depth=5, top_k=20,
    # logits_processor=logits_processor,
    constraints=constraints
)
print_output(input, output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


init constraint <constraints.constraint.ClassifiableConstraint object at 0x31d1538b0> seqlen 3
gpt: forward request text: |When we want to go| k 20 depth 5 beams 18 beam groups 3 logits [] consraints [<constraints.constraint.ClassifiableConstraint object at 0x31d1538b0>]
{'input_ids': tensor([[2215,  356,  765,  284,  467]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]]), 'offset_mapping': tensor([[[ 0,  4],
         [ 4,  7],
         [ 7, 12],
         [12, 15],
         [15, 18]]])}
Attention mask tensor([[1, 1, 1, 1, 1]])
gpt: generating with num_beams 18 num_return_sequences 18 num_beam_groups 3 max len 10
init constraint <constraints.constraint.ClassifiableConstraint object at 0x31d153670> seqlen 3
copying stateful False WARNING THIS IS FALSE IN THE GIVEN IMPLEMENTATION??
init constraint <constraints.constraint.ClassifiableConstraint object at 0x31d1534c0> seqlen 3
copying stateful False WARNING THIS IS FALSE IN THE GIVEN IMPLEMENTATION??
init constraint <constraints.constraint.Cla

## Test search with constraint implementation

In [58]:

# force_words = [[" rice", " beans", " cheese", "play", "find", "ize", "ful"]]
# constraints = []
# for constraint_entitiy in force_words:
#     tokens = tokenizer(constraint_entitiy).input_ids
#     print('constraint tokens', tokens)
#     constraints.append(DisjunctiveConstraint(tokens))


force_words = ["rice", "beans", "cheese", "rice", " playfulness"]

word_ids = []
for word in force_words:
    tokens = tokenizer(word).input_ids
    print('word', word, tokens)
    word_ids.append(tokens)

constraint = DisjunctiveConstraint([tokens])

input = "I love to cook"
output = forward_search(
    input, depth=10, top_k=10,
    # logits_processor=logits_processor,
    constraints=[constraints]
)
print_output(input, output)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


word rice [20970]
word beans [44749]
word cheese [2395, 2771]
word rice [20970]
word playfulness [1759, 15538]
gpt: forward request text: |I love to cook| k 10 depth 10 beams 9 beam groups 3 logits [] consraints [[<constraints.constraint.ClassifiableConstraint object at 0x152ddc850>]]
{'input_ids': tensor([[  40, 1842,  284, 4255]]), 'attention_mask': tensor([[1, 1, 1, 1]]), 'offset_mapping': tensor([[[ 0,  1],
         [ 1,  6],
         [ 6,  9],
         [ 9, 14]]])}
gpt: generating with num_beams 9 num_return_sequences 9 num_beam_groups 3 max len 14


AttributeError: 'list' object has no attribute 'seqlen'

In [89]:
[i for i in range(len(['a', 'b', 'c']) -1)]

[0, 1]

In [39]:
import constraints.constraint
from importlib import reload
reload(constraints.constraint)
from constraints.constraint import *

In [40]:
test()

init constraint <constraints.constraint.ClassifiableConstraint object at 0x152d5b280>
calling update with token 464
does_advance called in Constraint contains, token 464 for sequence []
does_advance True
checker tokens ['The'], ['DET'], expected ['DET', 'DET', 'DET'] completed False
Token: The, Stepped: True, Completed: False, Reset: False
calling update with token 3797
does_advance called in Constraint contains, token 3797 for sequence [464]
does_advance True
checker tokens ['The', ' cat'], ['DET', 'NOUN'], expected ['DET', 'DET', 'DET'] completed False
Token:  cat, Stepped: True, Completed: False, Reset: False
calling update with token 318
does_advance called in Constraint contains, token 318 for sequence [464, 3797]
does_advance True
checker tokens ['The', ' cat', ' is'], ['DET', 'NOUN', 'VERB'], expected ['DET', 'DET', 'DET'] completed False
Token:  is, Stepped: True, Completed: False, Reset: False
calling update with token 319
does_advance called in Constraint contains, token 319 

# Logits Processor Experiments


In [33]:
import tensorflow as tf
from transformers import TFLogitsProcessor

class ConstraintLogitsProcessor(TFLogitsProcessor):
    r"""
    [`TFLogitsProcessor`] that boosts tokens that sequentially advance towards a predefined list of target encoded tokens.
    
    This processor dynamically adjusts the generation scores to favor tokens that are the next correct sequential continuation of a target phrase like "I love Alex so very much", starting from the most recent token generated in the sequence.
    """

    def __init__(self, target, tokenizer):
        super().__init__()
        self.target_tokens = tokenizer.encode(target)
        print('target_tokens', self.target_tokens)

        self.boost_factor = 10
        self.target_indices = None  # Will be initialized with batch size on first call

    def __call__(self, input_ids: tf.Tensor, scores: tf.Tensor, cur_len: int) -> tf.Tensor:
        batch_size, num_tokens = scores.shape

        # Initialize target indices for each sequence in the batch if not already done
        if self.target_indices is None:
            self.target_indices = tf.zeros([batch_size], dtype=tf.int32)

        # Determine the last generated token for each sequence in the batch using cur_len
        last_token_ids = input_ids[:, cur_len - 1]

        # Apply the matching logic
        scores = self.apply_matching_logic(last_token_ids, scores)

        return scores

    def apply_matching_logic(self, last_token_ids, scores):
        indices_to_update = []
        updates = []

        # Collect updates in a list to apply them all at once later
        for idx in tf.range(tf.size(last_token_ids)):
            if self.target_indices[idx] < len(self.target_tokens):
                expected_token = self.target_tokens[self.target_indices[idx]] # this is what we would update based on our compiled list
                # Check if the last token matches the expected token in the target sequence
                if last_token_ids[idx] == expected_token:
                    # Boost the score for the next target token, if it exists in the sequence
                    next_idx = self.target_indices[idx] + 1
                    if next_idx < len(self.target_tokens):
                        next_target_token = self.target_tokens[next_idx]
                        indices_to_update.append([idx, next_target_token])
                        updates.append(self.boost_factor)

                    # Update the target index for this sequence
                    self.target_indices = tf.tensor_scatter_nd_update(
                        self.target_indices, [[idx]], [next_idx])

        # Apply all score updates at once
        if indices_to_update:
            scores = tf.tensor_scatter_nd_add(scores, indices_to_update, updates)

        return scores

target = "you can't clean yourself without cleaning your toilet"
# Create the LogitsProcessors
space_aware_processor = SpaceAwareLogitsProcessor(tokenizer);
endless_processor = EndlessLogitsProcessor(tokenizer)
constraint_logits_processor = ConstraintLogitsProcessor(target, tokenizer)
logits_processor = TFLogitsProcessorList([
    space_aware_processor, 
    endless_processor,
    constraint_logits_processor
    ]
)
input = "I like mountains "
output = forward_search(input, top_k=10, depth=18, logits_processor=logits_processor)
print_output(input, output)

target_tokens [5832, 460, 470, 3424, 3511, 1231, 12724, 534, 16146]
forward |I like mountains| k 10 depth 18 beams 9 beam groups 3 ends space True
{'input_ids': <tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[   40,   588, 12269]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(1, 3), dtype=int32, numpy=array([[1, 1, 1]], dtype=int32)>, 'offset_mapping': <tf.Tensor: shape=(1, 3, 2), dtype=int32, numpy=
array([[[ 0,  1],
        [ 1,  6],
        [ 6, 16]]], dtype=int32)>}
ends with space True input_len 3 max_length 21
Shape of beam_token_scores: [TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape([9, 50257]), TensorShape(

# Spacy loading experiments

In [36]:

import spacy
import time

# Load spaCy's English language model
nlp = spacy.load("en_core_web_sm")

# Example texts of various lengths
texts = [
    "Quick brown fox jumps over the lazy dog.",
    "In project management, a project consists of a temporary endeavor undertaken to create a unique product, service, or result.",
    "The importance of sustainable energy has been recognized on a global scale, with multiple countries investing heavily in solar, wind, and hydroelectric power generation systems.",
    "Neural networks, a subset of machine learning algorithms, have revolutionized the field of artificial intelligence by enabling computers to perform complex tasks such as image recognition, natural language processing, and prediction based on large datasets.",
    "Recent advances in quantum computing have opened up new possibilities for solving problems that were previously thought to be intractable with classical computers. This includes complex molecular modeling for drug discovery, optimization problems in logistics and supply chains, and the development of new materials with designed properties."
]

texts.append(texts[-1] * 10)  # Add a longer text for more comprehensive testing

# Function to measure the time taken to tokenize text and calculate time per word
def time_tokenization(text):
    start_time = time.time()
    doc = nlp(text)
    duration = time.time() - start_time
    num_words = len(doc)
    time_per_word = duration / num_words if num_words else 0
    return num_words, duration, time_per_word

# Measure tokenization times
results = []
for text in texts:
    num_words, total_time, time_per_word = time_tokenization(text)
    results.append((num_words, total_time, time_per_word))

# Calculate the average time per word from the collected data
average_time_per_word = sum(time_per_word for _, _, time_per_word in results) / len(results)
worst_case_time = max(time_per_word for _, _, time_per_word in results)

# Function to estimate the time for a given number of words
def estimate_time_for_words(word_count, time_per_word):
    estimated_time = word_count * time_per_word
    return estimated_time

# Input: Number of words to estimate time for
desired_word_count = int(input("Enter the number of words to estimate tokenization time for: "))

# Calculate estimated time
estimated_time = estimate_time_for_words(desired_word_count, average_time_per_word)
estimated_time_worst_case = estimate_time_for_words(desired_word_count, worst_case_time)
print(f"Estimated time to tokenize {desired_word_count} words: {estimated_time:.4f} seconds")
print(f"Worst-case estimated time: {estimated_time_worst_case:.4f} seconds")

# Display results from initial tests for reference
for num_words, total_time, time_per_word in results:
    print(f"Words: {num_words}, Total time: {total_time:.4f} seconds, Time per word: {time_per_word:.4f} seconds")

TypeError: 'str' object is not callable